<h3>Cấu hình

In [25]:
# Thư viện from pathlib import Path
import sqlite3
import pandas as pd
import warnings
import duckdb
import gc
from pathlib import Path

# Tắt warning datavalidation khi đọc file Excel
warnings.filterwarnings("ignore", category=UserWarning, module="openpyxl")

In [26]:
# Mapping cột với tên gốc, datatype,...
RAW_DATA_MAPPING = { "order_no": {"keywords": ["STT"], "dtype": "string", "use": True},
    "order_code": {
        "keywords": ["Mã đơn hàng"],
        "dtype": "string",
        "use": True,
    },
    "region": {"keywords": ["Miền"], "dtype": "string", "use": False},
    "province": {
        "keywords": ["Tỉnh"],
        "dtype": "string",
        "use": False,
    },
    "provider_code": {
        "keywords": ["Mã NCC (Site nguồn)"],
        "dtype": "string",
        "use": True,
    },
    "provider_name": {
        "keywords": ["Tên NCC"],
        "dtype": "string",
        "use": False,
    },
    "provider_address": {
        "keywords": ["Địa chỉ NCC"],
        "dtype": "string",
        "use": False,
    },
    "subrange_code": {
        "keywords": ["Mã Sub Range"],
        "dtype": "string",
        "use": False,
    },
    "subrange_name": {
        "keywords": ["Tên Sub Range"],
        "dtype": "string",
        "use": False,
    },
    "location_code": {
        "keywords": ["Mã điểm giao"],
        "dtype": "string",
        "use": True,
    },
    "location_name": {
        "keywords": ["Tên điểm giao"],
        "dtype": "string",
        "use": True,
    },
    "location_address": {
        "keywords": ["Địa chỉ điểm giao"],
        "dtype": "string",
        "use": True,
    },
    "buyer": {
        "keywords": ["Người đặt hàng"],
        "dtype": "string",
        "use": True,
    },
    "product_order": {
        "keywords": ["STT sản phẩm"],
        "dtype": "string",
        "use": True,
    },
    "product_code": {
        "keywords": ["Mã hàng"],
        "dtype": "string",
        "use": False,
    },
    "product_name": {
        "keywords": ["Tên hàng"],
        "dtype": "string",
        "use": True,
    },
    "barcode": {
        "keywords": ["Mã Barcode hàng hóa"],
        "dtype": "string",
        "use": True,
    },
    "uom": {
        "keywords": ["ĐVT"],
        "dtype": "string",
        "use": False,
    },
    "winmart_price": {
        "keywords": ["Đơn giá"],
        "dtype": "decimal",
        "use": True,
    },
    "product_quantity": {
        "keywords": ["Số lượng đặt hàng"],
        "dtype": "decimal",
        "use": True,
    },
    "weight": {
        "keywords": ["Trọng lượng (KG)"],
        "dtype": "decimal",
        "use": False,
    },
    "promised_quantity": {
        "keywords": ["Số lượng hẹn giao hàng"],
        "dtype": "decimal",
        "use": False,
    },
    "shipped_quantity": {
        "keywords": ["SL NCC đã giao"],
        "dtype": "decimal",
        "use": False,
    },
    "actual_quantity": {
        "keywords": ["Số lượng thực tế giao hàng"],
        "dtype": "decimal",
        "use": False,
    },
    "order_date": {
        "keywords": ["Ngày đặt hàng"],
        "dtype": "date",
        "use": True
    },
    "demand_date": {
        "keywords": ["Ngày yêu cầu giao hàng"],
        "dtype": "date",
        "use": True,
    },
    "promised_date": {
        "keywords": ["Ngày NCC hẹn giao hàng"],
        "use": False,
    },
    "promised_time_slot": {
        "keywords": ["Khung giờ hẹn giao hàng"],
        "use": False,
    },
    "confirm_date": {
        "keywords": ["Ngày xác nhận giao hàng"],
        "use": False,
    },
    "confirm_time_slot": {
        "keywords": ["Khung giờ xác nhận giao hàng"],
        "use": False,
    },
    "delivery_date": {
        "keywords": ["Ngày giao hàng"],
        "use": False,
    },
    "supplier_confirm_date": {
        "keywords": ["Ngày NCC xác nhận đã giao"],
        "use": False,
    },
    "extension_date": {
        "keywords": ["Ngày gia hạn"],
        "use": False,
    },
    "status": {
        "keywords": ["Trạng thái"],
        "dtype": "string",
        "use": False,
    },
    "updated_at": {
        "keywords": ["Ngày cập nhật"],
        "use": False,
    },
    "note_1": {
        "keywords": ["Chú thích 1"],
        "dtype": "string",
        "use": False,
    },
    "note_2": {
        "keywords": ["Chú thích 2"],
        "dtype": "string",
        "use": False,
    },
}

DATA_FOLDER = Path("data")

ORDER_FOLDER = Path(DATA_FOLDER/"orders")
DB_PATH = Path("database.db")
INPUT_FILE = Path(DATA_FOLDER/"input_winmart.xlsx")

In [27]:
def read_order_file():
    """
    Hàm này lặp qua tất cả các file trong thư mục chứa đơn hàng và trả về 1 dataframe
    """
    dfs = []
    for f in ORDER_FOLDER.glob("*.xlsx"):
        df = pd.read_excel(f)
        print(f"Đang đọc file {f.name}")

        # Đổi tên cột
        rename_dict = {v['keywords'][0]:k for k, v in RAW_DATA_MAPPING.items() if v['use'] == True }
        df_rename = df.rename(columns=rename_dict)

        # Drop các cột không cần thiết
        select_cols = [col for col in rename_dict.values()]
        df_select_cols = df_rename[select_cols].copy()

        # Đổi định dạng
        dtype_map = {
            k: v["dtype"]
            for k, v in RAW_DATA_MAPPING.items()
            if v.get("use") and "dtype" in v
        }

        for col, dtype in dtype_map.items():
            if dtype == "date":
                df_select_cols[col] = pd.to_datetime(df_select_cols[col], format="%d/%m/%Y")
            elif dtype == "decimal":
                df_select_cols[col] = pd.to_numeric(df_select_cols[col], errors='coerce')
            else:
                df_select_cols[col] = df_select_cols[col].astype("string")

        dfs.append(df_select_cols)
    return pd.concat(dfs)


def read_input_file(sheet):
    """
    Đọc các sheet excel từ file input_winmart.xlsx
    Trả về DataFrame
    """
    table_dict = {
        "product": "Sản phẩm",
        "product_price": "Giá gốc",
        "promotion": "Chương trình khuyến mãi",
        "location": "Thông tin điểm giao",
        "customer": "Khách hàng",
        "cost_center": "Cost Center"
    }
    df = pd.read_excel(
        DATA_FOLDER / "input_winmart.xlsx",
        sheet_name=table_dict[sheet]
    )
    return df


def execute_sql(script):
    """
    Tạo bảng và constraint trong databse
    """
    global conn
    # Đóng connection cũ nếu tồn tại
    try:
        conn.close()
    except:
        pass

    try:
        del conn
    except:
        pass

    gc.collect()

    if DB_PATH.exists():
        DB_PATH.unlink()
    with sqlite3.connect(DB_PATH) as conn:
        conn.execute("PRAGMA foreign_key = ON")
        with open(script, "r", encoding="utf-8") as f:
            sql = f.read()

        conn.executescript(sql)
        conn.commit()


def import_dtb(df, tablename):
    """
    Làm sạch dữ liệu trong bảng trước khi import
    Chỉ import data từ DataFrame, không phá các constraint của bảng
    Các cột trong DataFrame không cần đúng thứ tự, chỉ cần tên giống nhau
    """
    with sqlite3.connect("database.db") as conn:
        conn.execute("PRAGMA foreign_key = ON")
        cursor = conn.cursor()
        cursor.execute(f"DELETE FROM {tablename}")
        df.to_sql(
            tablename,
            index=False,
            con=conn,
            if_exists="append"
        )
        conn.commit()


def show_result(DB_PATH):
    """
    Trả về một DataFrame chứa kết qủa import
    """
    with sqlite3.connect(DB_PATH) as conn:
        tables = pd.read_sql("""
            SELECT name
            FROM sqlite_master
            WHERE type = 'table'
            AND name NOT LIKE 'sqlite_%'
        """, conn)

        result = []

        for table in tables["name"]:
            query = f"""
                SELECT COUNT(*) AS row_count
                FROM {table}
            """
            count = pd.read_sql(query, conn).iloc[0, 0]
            result.append({
                "table_name": table,
                "row_count": count
            })
    return pd.DataFrame(result)

In [28]:
#Tạo các bảng và constraint
execute_sql("sql_scripts.sql")

In [29]:
# Đọc dữ liệu đơn hàng
import_order = read_order_file()
# Import vào database
import_dtb(import_order, "stg_transactions")

Đang đọc file ĐH 22.5.26.xlsx


In [30]:
# Đọc sheet sản phẩm
product = read_input_file("product")

# Tách nhóm sản phẩm khỏi bảng sản phẩm và mport
import_product_category = duckdb.sql("""
    SELECT
        ROW_NUMBER() OVER() AS product_category_key,
        "MSG - DS - NK - ĐV" AS ctg_1,
        "DS - ĐV" AS ctg_2
    FROM (
        SELECT DISTINCT
            "DS - ĐV",
            "MSG - DS - NK - ĐV"
        FROM product
    )
""").to_df()

# Import dữ liệu vào bảng product_category
import_dtb(import_product_category, "product_category")

import_product = duckdb.sql("""
    SELECT
        CAST("Mã sản phẩm" as TEXT) AS product_code,
        "Tên sản phẩm" AS product_name,
        product_category_key
    FROM product pr
    LEFT JOIN import_product_category prc ON pr."MSG - DS - NK - ĐV" = prc.ctg_1
""").to_df()

# Import dữ liệu vào bảng product
import_dtb(import_product, "product")

In [31]:
# Đọc sheet Giá gốc
base_price = read_input_file( "product_price")

# Trích xuất dữ liệu loai hệ thống (win+ / winmart) từ bảng giá gốc
import_customer_system = duckdb.sql("""
    SELECT
        ROW_NUMBER() OVER() AS system_key,
        "Hệ thống" AS system_name
    FROM (
         SELECT DISTINCT "Hệ thống"
         FROM base_price
         WHERE "Hệ thống" IS NOT NULL
    )
""").to_df()
# Import dữ liệu vào customer_system
import_dtb(import_customer_system, "customer_system")

# Lấy dữ liệu của bảng giá gốc
import_base_price = duckdb.sql("""
    SELECT DISTINCT
        CAST("Mã Barcode" AS TEXT) AS barcode,
        CAST("Mã sản phẩm" AS TEXT) AS product_code,
        CAST("Giá gốc" AS numeric) AS base_price
    FROM
        base_price bpr
""").to_df()

# Import giá gốc vào database
import_dtb(import_base_price, "base_price")

In [32]:
# Đọc dữ liệu từ sheet "Chương trình khuyến mãi"
promotion = read_input_file("promotion")
import_promo_type = duckdb.sql("""
    SELECT
        ROW_NUMBER() OVER() AS promo_type_key,
        "Chương trình KM" AS promo_type_name
    FROM (
        SELECT DISTINCT "Chương trình KM"
        FROM promotion
        WHERE "Chương trình KM" IS NOT NULL
    )
""").to_df()
import_dtb(import_promo_type, "promo_type")

# Chi tiết chương trình khuyến mãi
import_promotion_detail = duckdb.sql("""
    WITH pr AS(
        SELECT
            CAST("Tên chương trình" AS TEXT) AS post_name,
            CAST("Hệ thống" AS TEXT) AS system_name,
            CAST(strptime("Ngày áp dụng", '%d/%m/%Y') AS DATE) AS start_date,
            CAST(strptime("Ngày kết thúc", '%d/%m/%Y')  AS DATE) AS end_date,
            CAST("Barcode" AS TEXT) AS barcode,
            CAST("% Giảm giá" AS DOUBLE) AS discount_percentage,
            CAST("Chương trình KM" AS TEXT) AS promo_type_name,
            CAST("Mã hàng tặng" AS TEXT) AS promo_product_name
        FROM promotion
    )
    SELECT
        post_name,
        system_key,
        start_date,
        end_date,
        discount_percentage,
        promo_type_key,
        barcode
    FROM pr
    LEFT JOIN import_customer_system USING (system_name)
    LEFT JOIN import_promo_type USING (promo_type_name)
""").to_df()
import_dtb(import_promotion_detail, "promotion_detail")

In [33]:
# Đọc dữ liệu từ sheet Khách hàng
customer = read_input_file( "customer")

# Lấy dữ liệu mã kho và import
import_warehouse = duckdb.sql("""
    SELECT
        ROW_NUMBER() OVER() AS warehouse_key,
        "Mã kho" AS warehouse_code,
    FROM (
        SELECT DISTINCT "Mã kho"
        FROM customer
         )
    customer
""").to_df()
import_dtb(import_warehouse, "warehouse")

# Lấy dữ liệu chi nhánh và import
import_branch = duckdb.sql("""
    SELECT
        ROW_NUMBER() OVER() AS branch_key,
        "Chi nhánh" AS branch_name,
    FROM (
        SELECT DISTINCT "Chi nhánh"
        FROM customer
         )
    customer
""").to_df()
import_dtb(import_branch, "branch")

# Join lại dữ liệu 3 bảng dim (warehouse, branch, product_category) vào customer
import_customer = duckdb.sql("""
    WITH customer_r AS (
        SELECT CAST("Mã khách hàng" AS TEXT) AS customer_code,
        "Tên khách hàng" AS customer_name,
        "Chi nhánh" AS branch_name,
        "Nhóm hàng" AS ctg_1,
        "Mã kho" AS warehouse_code
        FROM customer
    )
    SELECT
    -- *,
        customer_code,
        customer_name,
        branch_key,
        warehouse_key,
        product_category_key
    FROM customer_r r
    LEFT JOIN import_warehouse USING (warehouse_code)
    LEFT JOIN import_branch USING (branch_name)
    LEFT JOIN import_product_category USING (ctg_1)
""").to_df()
import_dtb(import_customer, "customer")

In [34]:
# Đọc dữ liệu từ sheet Thông tin điểm giao
location = read_input_file( "location")

# Đọc dữ liệu và import vào bảng location
import_location = duckdb.sql("""
    WITH location_r AS (
        SELECT
            "Mã điểm giao" AS location_code,
            "Hệ thống" AS system_name,
            "Tên điểm giao" AS location_name,
            "Địa chỉ điểm giao" AS location_address,
            "Mã kho" AS warehouse_code,
            "Nhóm hàng" AS ctg_1,
            "Mã khách hàng" AS customer_code
        FROM location
    )
    SELECT
    location_code,
    location_address,
    system_key,
    customer_code
    FROM location_r
    LEFT JOIN import_customer_system USING (system_name)
    LEFT JOIN import_warehouse USING (warehouse_code)
    LEFT JOIN import_product_category USING(ctg_1)
    --WHERE product_category_key IS NULL
""").to_df()
import_dtb(import_location, "delivery_location")

Bảng Cost Center

In [35]:
# Đọc sheet Cost Center
cost_center = read_input_file( "cost_center")

import_cost_center = duckdb.sql("""
    WITH cost_center_r AS (
        SELECT
        "Nhóm sản phẩm" AS ctg_1,
        "Chi nhánh" AS branch_name,
        "Cost center" AS cost_center_code,
        "Tên" AS cost_center_name
        FROM cost_center
    )
    SELECT
        cost_center_code,
        cost_center_name,
        branch_key,
        product_category_key
    FROM cost_center_r
    LEFT JOIN import_product_category USING (ctg_1)
    LEFT JOIN import_branch USING (branch_name)
""").to_df()
import_dtb(import_cost_center, "cost_center")

<h3> Kết quả

In [36]:
show_result(DB_PATH)

,table_name,row_count
0,product_category,4
1,product,976
2,customer_system,3
3,warehouse,6
4,branch,5
5,customer,49
6,delivery_location,3764
7,promo_type,2
8,base_price,171
9,promotion_detail,222
